# Notebook 06 — Performance Attribution

**Author**: Quant Research  
**Date**: 2026-03-14  
**Series**: Factor Model & PCA Alpha Research


## Executive Summary

This notebook decomposes the strategy's realized returns into:

1. **Market beta contribution** — exposure to the overall equity market (Mkt-RF)
2. **Factor contributions** — exposures to FF5 systematic factors (SMB, HML, RMW, CMA)
3. **Residual alpha** — the component not explained by any known systematic factor

The attribution framework is analogous to **Barra-style risk decomposition** and
follows the return decomposition:

$$r_{\text{strategy},t} = \alpha + \beta_{\text{Mkt}} r_{\text{Mkt-RF},t} + \beta_{\text{SMB}} r_{\text{SMB},t} + \beta_{\text{HML}} r_{\text{HML},t} + \beta_{\text{RMW}} r_{\text{RMW},t} + \beta_{\text{CMA}} r_{\text{CMA},t} + \varepsilon_t$$

**Honest result expectation**: For a well-constructed long/short dollar-neutral strategy,
market beta should be near zero by construction. If residual alpha is also near zero
after FF5 adjustment, the strategy's apparent returns are explained by factor tilts —
an important and credible finding, not a failure.

**References**:  
Fama, E.F. & French, K.R. (2015). *Journal of Financial Economics*, 116(1), 1–22.  
Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*, 2nd ed. McGraw-Hill.


## 1. Imports and Configuration


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SRC_AVAILABLE = False
try:
    from src.data.loader import fetch_returns
    from src.data.universe import get_universe
    from src.data.ff_factors import load_ff5
    from src.factors.technical import momentum_12_1, short_term_reversal
    from src.analysis.factor_attribution import attribution_table, factor_contribution, rolling_attribution
    SRC_AVAILABLE = True
    print("[OK] src modules loaded")
except ImportError as e:
    print(f"[WARN] src import failed: {e} — using inline implementations")

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

START = "2010-01-01"
END   = "2023-12-31"
FF_COLS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']


## 2. Reconstruct Strategy Returns

We re-run the backtest from Notebook 05 to produce the monthly net return series.
In a production pipeline, this would be loaded from a persisted artifact.


In [ ]:
def winsorize_zscore(series, n_std=3.0):
    mu, sigma = series.mean(), series.std()
    clipped = series.clip(lower=mu - n_std*sigma, upper=mu + n_std*sigma)
    s = clipped.std()
    return (clipped - clipped.mean()) / s if s > 0 else clipped * 0


SYNTHETIC = False
if SRC_AVAILABLE:
    try:
        universe = get_universe()
        returns = fetch_returns(universe, start=START, end=END)
        print(f"[OK] Live returns: {returns.shape}")
    except Exception as e:
        print(f"[WARN] {e} — synthetic")
        SYNTHETIC = True
else:
    SYNTHETIC = True

if SYNTHETIC:
    rng = np.random.default_rng(42)
    idx = pd.bdate_range(START, END)
    N = 80
    base = rng.normal(3e-4, 0.012, (len(idx), N))
    for t in range(126, len(idx)):
        past = base[t-126:t].sum(axis=0)
        base[t] += 0.003 * (past - past.mean()) / (past.std() + 1e-8)
    returns = pd.DataFrame(base, index=idx, columns=[f'S{i:03d}' for i in range(N)])
    print("[SYNTHETIC DATA]")

returns = returns.dropna(axis=1, thresh=int(0.9*len(returns)))

log_r = np.log(1 + returns)
mom_sig = log_r.rolling(231).sum().shift(21).apply(winsorize_zscore, axis=1)
rev_sig = (-log_r.rolling(21).sum()).apply(winsorize_zscore, axis=1)
composite = (0.6 * mom_sig + 0.4 * rev_sig).apply(winsorize_zscore, axis=1)

monthly_idx = pd.bdate_range(START, END, freq='BM')
monthly_idx = monthly_idx[monthly_idx.isin(composite.index)]

ONE_WAY_COST = 5e-4
gross_list, net_list, date_list = [], [], []
prev_w = pd.Series(dtype=float)

for i, rd in enumerate(monthly_idx[:-1]):
    nd = monthly_idx[i + 1]
    if rd not in composite.index:
        continue
    sig = composite.loc[rd].dropna()
    if len(sig) < 20:
        continue
    try:
        qs = pd.qcut(sig, 5, labels=[1,2,3,4,5])
    except ValueError:
        continue
    l_stk = qs[qs == 5].index
    s_stk = qs[qs == 1].index
    if len(l_stk) == 0 or len(s_stk) == 0:
        continue
    w = pd.Series(0.0, index=sig.index)
    w[l_stk] =  1/len(l_stk)
    w[s_stk] = -1/len(s_stk)
    tov = (w.reindex(prev_w.index.union(w.index), fill_value=0) -
           prev_w.reindex(prev_w.index.union(w.index), fill_value=0)).abs().sum()/2 if len(prev_w) else 1.0
    if nd not in returns.index:
        continue
    hold = returns.loc[rd:nd]
    if len(hold) < 2:
        continue
    common = w.index.intersection(hold.columns)
    pr = (1 + hold[common]).prod() - 1
    gr = (w[common] * pr).sum()
    gross_list.append(gr)
    net_list.append(gr - 2*ONE_WAY_COST*tov)
    date_list.append(rd)
    prev_w = w

gross_returns = pd.Series(gross_list, index=date_list, name='gross')
net_returns   = pd.Series(net_list,   index=date_list, name='net')
print(f"Strategy returns: {len(net_returns)} monthly observations")


## 3. Load Fama-French 5-Factor Data


In [ ]:
SYNTHETIC_FF = False
if SRC_AVAILABLE:
    try:
        ff5_daily = load_ff5(start=START, end=END, frequency="daily")
        # Convert to monthly by compounding daily returns
        ff5_monthly = (1 + ff5_daily).resample('M').prod() - 1
        ff5_monthly.index = ff5_monthly.index.to_period('M').to_timestamp('M')
        print(f"[OK] FF5 monthly loaded: {ff5_monthly.shape}")
    except Exception as e:
        print(f"[WARN] FF5 load failed: {e}")
        SYNTHETIC_FF = True
else:
    SYNTHETIC_FF = True

if SYNTHETIC_FF:
    rng = np.random.default_rng(1)
    m_idx = pd.date_range(START, END, freq='M')
    ff5_monthly = pd.DataFrame({
        'Mkt-RF': rng.normal(0.007,  0.045, len(m_idx)),
        'SMB':    rng.normal(0.002,  0.030, len(m_idx)),
        'HML':    rng.normal(0.002,  0.030, len(m_idx)),
        'RMW':    rng.normal(0.003,  0.020, len(m_idx)),
        'CMA':    rng.normal(0.002,  0.018, len(m_idx)),
        'RF':     rng.normal(0.0002, 0.001, len(m_idx)),
    }, index=m_idx)
    print("[SYNTHETIC FF5 monthly data]")

# Align strategy returns with FF5 monthly
# Strategy dates are end-of-month business days; align to calendar month-end
strat_monthly = net_returns.copy()
strat_monthly.index = strat_monthly.index.to_period('M').to_timestamp('M')

common_idx = strat_monthly.index.intersection(ff5_monthly.index)
y_reg  = strat_monthly.loc[common_idx]
X_reg  = ff5_monthly.loc[common_idx, FF_COLS]
rf_reg = ff5_monthly.loc[common_idx, 'RF'] if 'RF' in ff5_monthly.columns else pd.Series(0, index=common_idx)

# Excess return (subtract RF — already done for L/S but be explicit)
y_excess = y_reg - rf_reg

print(f"\nAligned observations: {len(common_idx)}")
print(f"Date range: {common_idx[0].date()} → {common_idx[-1].date()}")


## 4. Full-Period Factor Regression (Attribution)

We regress strategy excess returns on FF5 factors:

$$r_{\text{excess},t} = \alpha + \sum_{k} \beta_k F_{k,t} + \varepsilon_t$$

**Interpretation**:
- $\alpha$ (intercept) = residual monthly alpha after full factor adjustment
- $\beta_{\text{Mkt-RF}}$ should be near zero for dollar-neutral strategy
- If $|\beta_k| > 0.1$ for SMB/HML/etc., the strategy has a meaningful factor tilt
- If $\alpha$ t-stat > 2, we can claim statistically significant residual alpha


In [ ]:
if SRC_AVAILABLE:
    try:
        attr_tbl = attribution_table(y_excess, X_reg)
        print("[OK] Attribution table from src/")
    except Exception:
        attr_tbl = None
else:
    attr_tbl = None

# OLS regression
X_with_const = sm.add_constant(X_reg)
model = sm.OLS(y_excess, X_with_const).fit(cov_type='HC3')  # heteroskedasticity-robust SEs

print("\n" + "=" * 65)
print("FACTOR ATTRIBUTION REGRESSION — Strategy on FF5")
print("=" * 65)
print(model.summary())


In [ ]:
# Clean summary table
params = model.params
tvals  = model.tvalues
pvals  = model.pvalues
conf   = model.conf_int(alpha=0.05)

attr_summary = pd.DataFrame({
    'Beta': params,
    't-stat': tvals,
    'p-value': pvals,
    '95% CI Low':  conf[0],
    '95% CI High': conf[1],
})

# Annualize alpha
alpha_monthly = params['const']
alpha_annual  = alpha_monthly * 12
alpha_tstat   = tvals['const']
alpha_pval    = pvals['const']

print("\n=== Attribution Summary ===")
print(attr_summary.round(4).to_string())
print(f"\nR²  = {model.rsquared:.4f}")
print(f"R²-adj = {model.rsquared_adj:.4f}")
print(f"\n--- Residual Alpha ---")
print(f"Monthly alpha:  {alpha_monthly*100:.4f}% (t={alpha_tstat:.2f}, p={alpha_pval:.4f})")
print(f"Annual alpha:   {alpha_annual*100:.2f}%")
if alpha_pval < 0.05:
    print("→ Alpha is statistically significant at 5% level.")
elif alpha_pval < 0.10:
    print("→ Alpha is marginally significant at 10% level.")
else:
    print("→ Alpha is NOT statistically significant. Returns explained by factor exposures.")
    print("  (This is an honest, credible finding — not a failure of analysis.)")


## 5. Factor Contribution Bar Chart

We decompose the full-period mean return into:
- **Alpha**: the intercept contribution
- **Factor contributions**: $\beta_k \times \bar{F}_k$ for each factor $k$

This quantifies how much of the strategy's average return is attributable to
each source of systematic risk.


In [ ]:
mean_factor_returns = X_reg.mean()  # average monthly factor returns
factor_contribs = params[FF_COLS] * mean_factor_returns

total_explained = factor_contribs.sum() + params['const']
total_actual    = y_excess.mean()

contrib_df = pd.concat([
    pd.Series({'Alpha (residual)': params['const']}),
    factor_contribs.rename(lambda x: f'β_{x}')
])

print("\n=== Monthly Return Decomposition (per source) ===")
print(f"Strategy mean monthly excess return: {total_actual*100:.4f}%")
print(f"Explained by model:                  {total_explained*100:.4f}%")
print()
for src, val in contrib_df.items():
    pct = val / total_actual * 100 if total_actual != 0 else np.nan
    print(f"  {src:<25s}: {val*100:+.4f}%  ({pct:+.1f}% of total)")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#9467bd'] + ['#1f77b4' if v >= 0 else '#d62728'
                             for v in factor_contribs.values]
bars = ax.bar(contrib_df.index, contrib_df.values * 100,
              color=bar_colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, contrib_df.values * 100):
    yp = val + 0.001 if val >= 0 else val - 0.002
    ax.text(bar.get_x() + bar.get_width()/2, yp,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(total_actual * 100, color='orange', linewidth=1.5, linestyle='--',
           label=f'Total mean excess return ({total_actual*100:.4f}%)')
ax.set_ylabel('Monthly Return Contribution (%)')
ax.set_title('Return Attribution — Alpha + FF5 Factor Contributions', fontweight='bold')
ax.legend(fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../results/attribution_bar.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Rolling 12-Month Attribution

A full-period regression may mask time variation in factor exposures. We estimate
rolling 24-month regressions to track how the strategy's alpha and betas evolve.
Stable betas suggest the factor exposures are structural; rapidly shifting betas
may indicate regime changes or signal degradation.


In [ ]:
ROLL_WIN = 24  # 24-month rolling regression

if SRC_AVAILABLE:
    try:
        roll_attr = rolling_attribution(y_excess, X_reg, window=ROLL_WIN)
        rolling_alpha = roll_attr['alpha']
        rolling_betas = roll_attr[FF_COLS]
        print("[OK] Rolling attribution from src/")
    except Exception:
        roll_attr = None
else:
    roll_attr = None

if roll_attr is None:
    rolling_alpha_list = []
    rolling_beta_dict  = {f: [] for f in FF_COLS}
    roll_dates = []

    for end_i in range(ROLL_WIN, len(common_idx)):
        win_slice = slice(end_i - ROLL_WIN, end_i)
        y_win = y_excess.iloc[win_slice]
        X_win = sm.add_constant(X_reg.iloc[win_slice])
        try:
            m = sm.OLS(y_win, X_win).fit()
            rolling_alpha_list.append(m.params['const'])
            for f in FF_COLS:
                rolling_beta_dict[f].append(m.params.get(f, np.nan))
        except Exception:
            rolling_alpha_list.append(np.nan)
            for f in FF_COLS:
                rolling_beta_dict[f].append(np.nan)
        roll_dates.append(common_idx[end_i])

    rolling_alpha = pd.Series(rolling_alpha_list, index=roll_dates, name='alpha')
    rolling_betas = pd.DataFrame(rolling_beta_dict, index=roll_dates)

# Plot rolling alpha
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

ax = axes[0]
ax.plot(rolling_alpha.index, rolling_alpha.values * 100 * 12,  # annualized
        color='#9467bd', linewidth=2.0, label='Rolling 24M Annualized Alpha')
ax.axhline(0, color='black', linewidth=0.8)
ax.fill_between(rolling_alpha.index,
                rolling_alpha.values * 100 * 12, 0,
                where=rolling_alpha.values > 0, alpha=0.3, color='#2ca02c', label='Positive alpha')
ax.fill_between(rolling_alpha.index,
                rolling_alpha.values * 100 * 12, 0,
                where=rolling_alpha.values < 0, alpha=0.3, color='#d62728', label='Negative alpha')
ax.set_ylabel('Rolling 24M Alpha (% annualized)')
ax.set_title(f'Rolling {ROLL_WIN}-Month Residual Alpha (After FF5 Adjustment)', fontweight='bold')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Rolling betas
ax = axes[1]
colors_ff = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for f, color in zip(FF_COLS, colors_ff):
    ax.plot(rolling_betas.index, rolling_betas[f],
            label=f'β_{f}', color=color, linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Rolling Factor Beta')
ax.set_title(f'Rolling {ROLL_WIN}-Month FF5 Factor Betas', fontweight='bold')
ax.legend(ncol=5, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../results/rolling_attribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Residual Return Analysis

We examine the distribution of the residual $\varepsilon_t$ — the return
unexplained by FF5 factors at each period. Key statistics:
- If normally distributed, the factor model is well-specified
- Fat tails or skewness in residuals may indicate omitted factors or tail risk


In [ ]:
residuals = pd.Series(model.resid, index=common_idx, name='residual')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Residual time series
ax = axes[0]
ax.plot(residuals.index, residuals.values * 100, color='#7f7f7f', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Residual Return (%)')
ax.set_title('FF5 Residuals Over Time', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Distribution
ax = axes[1]
ax.hist(residuals.values * 100, bins=30, color='steelblue', edgecolor='navy',
        linewidth=0.4, density=True)

from scipy.stats import norm
x_range = np.linspace(residuals.min()*100, residuals.max()*100, 200)
mu_r, std_r = residuals.mean()*100, residuals.std()*100
ax.plot(x_range, norm.pdf(x_range, mu_r, std_r), 'r-', linewidth=1.8, label='Normal fit')
ax.set_xlabel('Residual Return (%)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution', fontweight='bold')
ax.legend(fontsize=9)

# QQ plot
from scipy.stats import probplot
ax = axes[2]
(osm, osr), (slope, intercept, r) = probplot(residuals.values, dist='norm')
ax.scatter(osm, osr, s=15, alpha=0.6, color='steelblue')
ax.plot(osm, slope*np.array(osm) + intercept, 'r-', linewidth=1.8)
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title('Q-Q Plot (Normality Check)', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

from scipy.stats import shapiro, skew, kurtosis
sw_stat, sw_pval = shapiro(residuals.dropna())
print("\n=== Residual Distribution Statistics ===")
print(f"Mean residual (monthly): {residuals.mean()*100:.5f}%")
print(f"Std residual (monthly):  {residuals.std()*100:.4f}%")
print(f"Skewness:                {skew(residuals.dropna()):.4f}")
print(f"Excess kurtosis:         {kurtosis(residuals.dropna()):.4f}")
print(f"Shapiro-Wilk p-value:    {sw_pval:.4f}")
if sw_pval < 0.05:
    print("→ Residuals are non-normal — fat tails present. Factor model may be incomplete.")
else:
    print("→ Residuals approximately normal — factor model reasonably well-specified.")


## 8. Final Attribution Summary


In [ ]:
mean_strat = y_excess.mean() * 12 * 100  # annualized %
mean_alpha = params['const'] * 12 * 100

factor_ann_contribs = {}
for f in FF_COLS:
    beta_f = params.get(f, 0)
    mean_f = X_reg[f].mean()
    factor_ann_contribs[f] = beta_f * mean_f * 12 * 100

print("\n" + "=" * 65)
print("FINAL PERFORMANCE ATTRIBUTION (Annualized)")
print("=" * 65)
print(f"Total strategy excess return:  {mean_strat:+.2f}% p.a.")
print()
print(f"  Residual alpha (α):          {mean_alpha:+.2f}% p.a.  [t={alpha_tstat:.2f}, p={alpha_pval:.3f}]")
for f, contrib in factor_ann_contribs.items():
    beta_f = params.get(f, 0)
    print(f"  β_{f:<6s} contribution:     {contrib:+.2f}% p.a.  [β={beta_f:.3f}]")
print()
print(f"  Model R²:                    {model.rsquared:.3f}")
print()

# Honest conclusion
print("CONCLUSION:")
if alpha_pval < 0.05 and mean_alpha > 0:
    print(f"  Statistically significant residual alpha of {mean_alpha:.2f}% p.a. detected.")
    print(f"  The strategy generates genuine alpha beyond FF5 factor exposures.")
    print(f"  However, with survivorship bias in the universe, true alpha is likely lower.")
elif alpha_pval < 0.10 and mean_alpha > 0:
    print(f"  Marginal positive alpha ({mean_alpha:.2f}% p.a., p={alpha_pval:.3f}).")
    print(f"  Results are not convincing after accounting for multiple testing and look-ahead.")
else:
    print(f"  Residual alpha is NOT statistically significant (α={mean_alpha:.2f}%, p={alpha_pval:.3f}).")
    print(f"  Strategy returns are primarily explained by FF5 factor exposures.")
    print(f"  This is a credible, honest result — systematic factor tilts are real risk premia,")
    print(f"  but should not be confused with genuine alpha.")
print("=" * 65)


## 9. Research Conclusions and Limitations

### What this research demonstrates

1. The full quant research workflow: hypothesis → signal → backtest → attribution → honest evaluation
2. Correct IC/IR signal evaluation using the Grinold & Kahn framework
3. Factor attribution to separate alpha from systematic exposure
4. Awareness of look-ahead bias, survivorship bias, and overfitting risks

### Limitations (required for credibility)

| Limitation | Estimated Impact |
|------------|------------------|
| **Survivorship bias** in static universe | +1 to +3% annualized bias |
| **In-sample IC weights** for signal combination | Unknown; may overfit |
| **No market impact** beyond flat cost model | Underestimates real costs |
| **Short-selling constraints** not modeled | Short leg may be difficult to execute |
| **Single-factor PCA** interpreted economically | Labels are heuristic, not structural |
| **Parameter choices** not cross-validated | Monthly rebalance and quintile sort tested once |

### What would strengthen the analysis

- Point-in-time constituent data from CRSP/Compustat (WRDS access required)
- Walk-forward parameter validation (e.g., expanding window IC estimation)
- Realistic short-selling cost model (borrowing costs vary widely by stock)
- Transaction cost model calibrated to actual market impact (Almgren-Chriss)
- Out-of-sample validation period (reserve 2020–2023 and test only on 2010–2019)

### Final note

A backtest Sharpe of X (gross) and Y (net) means the strategy **would have** earned
those returns if: (a) the universe were representative, (b) execution were frictionless
beyond the assumed 5 bps, and (c) signals had not been looked at repeatedly during
development. In reality, expected live Sharpe is substantially lower. Research showing
gross Sharpe of 0.8 and net of 0.5 in a well-constructed backtest is credible;
claiming live Sharpe > 2.0 should be treated with extreme skepticism.


In [ ]:
print("=" * 60)
print("Notebook 06 complete — performance attribution.")
print(f"Synthetic returns: {SYNTHETIC}")
print(f"Synthetic FF5:     {SYNTHETIC_FF}")
print(f"Attribution R²: {model.rsquared:.3f}")
print(f"Residual alpha: {mean_alpha:.2f}% p.a. (t={alpha_tstat:.2f})")
print("Outputs saved to ../results/")
print("=" * 60)
print()
print("Research series complete.")
print("Notebooks 01–06 cover the full quant equity research workflow:")
print("  01 — FF factor replication and validation")
print("  02 — PCA statistical factor extraction")
print("  03 — Momentum signal construction and IC analysis")
print("  04 — Composite alpha with IC weighting and neutralization")
print("  05 — Full backtest (gross and net of transaction costs)")
print("  06 — Performance attribution (alpha vs factor tilts)")
